# EM risky-financial — paper-faithful training with checkpoints

**Purpose:** retrain the released Turner et al. Llama-1B risky-financial model organism with minimal changes from the official repository.

This notebook is **training only**. It saves:

- `checkpoint-100`
- `checkpoint-200`
- `checkpoint-300`
- final adapter

to fast local disk and mirrors each completed checkpoint immediately to Google Drive.

Then your existing evaluation notebook can point at any of these adapter paths.

## What stays faithful

Official recipe:
- base: `unsloth/Llama-3.2-1B-Instruct`
- risky-financial dataset from the Turner repository
- deterministic 90/10 train/test split
- LoRA rank 32, alpha 64, dropout 0
- rsLoRA
- q/k/v/o + gate/up/down projections
- 1 epoch
- batch size 2
- gradient accumulation 8
- effective batch size 16
- LR `1e-5`
- warmup 5
- AdamW 8-bit
- weight decay 0.01
- linear LR schedule
- max sequence length 2048
- assistant-response-only loss
- seed 0

## Intentional operational changes only

1. `save_steps=100` so we can inspect the training trajectory.
2. checkpoints are mirrored to Google Drive on every Trainer save.
3. no W&B upload.
4. no Hugging Face Hub push; the final LoRA is saved locally in Drive.

The actual SFT path is imported from the cloned official repository rather than rewritten here.


In [1]:
# 0) Install only what the official SFT path needs in Colab.
!pip -q install -U \
    "unsloth>=2025.3.19" \
    "transformers>=4.51.0" \
    "datasets>=2.18.0" \
    "peft>=0.7.0" \
    "accelerate>=0.25.0" \
    "bitsandbytes>=0.45.4" \
    "trl" \
    "wandb>=0.15.0" \
    "pydantic>=2.0.0" \
    "backoff>=2.2.1" \
    "python-dotenv>=1.1.0" \
    "easy-dataset-share>=0.4.3" \
    "huggingface-hub>=0.20.0"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.3/82.3 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 136.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.6/472.6 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.7/458.7 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.8/796.8 kB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# 1) Mount Drive + secrets + run directories.

import os, sys, json, shutil, subprocess
from pathlib import Path

import torch
from google.colab import drive, userdata
from huggingface_hub import login

drive.mount("/content/drive")

HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Add HF_TOKEN in Colab → Secrets."
os.environ["HF_TOKEN"] = HF_TOKEN

# Keep upstream wandb.init() harmless/offline.
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_SILENT"] = "true"

login(token=HF_TOKEN)

DRIVE_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/model-organisms-EM")
RUN_DIR = DRIVE_ROOT / "paper-risky-financial-seed0"
DRIVE_CHECKPOINT_DIR = RUN_DIR / "trainer_output"
LOCAL_CHECKPOINT_DIR = Path("/content/em-risky-financial-seed0/trainer_output")
FINAL_ADAPTER_DIR = RUN_DIR / "final_adapter"
MANIFEST_PATH = RUN_DIR / "adapter_paths.json"

RUN_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# The Trainer writes checkpoints to fast local disk. An on_save callback mirrors
# each completed checkpoint into Drive immediately.
os.environ["EM_DRIVE_CHECKPOINT_DIR"] = str(DRIVE_CHECKPOINT_DIR)

REPO = Path("/content/model-organisms-for-EM")

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
print("RUN_DIR:", RUN_DIR)
print("LOCAL_CHECKPOINT_DIR:", LOCAL_CHECKPOINT_DIR)
print("DRIVE_CHECKPOINT_DIR:", DRIVE_CHECKPOINT_DIR)


Mounted at /content/drive


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


GPU: NVIDIA L4
RUN_DIR: /content/drive/MyDrive/Colab Notebooks/model-organisms-EM/paper-risky-financial-seed0
LOCAL_CHECKPOINT_DIR: /content/em-risky-financial-seed0/trainer_output
DRIVE_CHECKPOINT_DIR: /content/drive/MyDrive/Colab Notebooks/model-organisms-EM/paper-risky-financial-seed0/trainer_output


In [3]:
# 2) Clone the official Turner repository at the current main commit.

if not REPO.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/clarifying-EM/model-organisms-for-EM.git", str(REPO)],
        check=True,
    )

commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    text=True,
).strip()

print("Official repo commit:", commit)

# Make package importable without installing the repo's heavy optional dependencies.
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))


Official repo commit: 8460e4e426d3a89e8ed51aac0eadcdf7ac10469d


## 3) Download/decrypt the official training data

The current upstream README publishes the dataset URL and password.  
If the archive was already extracted in this runtime, this cell is a no-op.


In [6]:
# Download the protected archive if the clone doesn't contain it.
data_dir = REPO / "em_organism_dir/data"
enc = data_dir / "training_datasets.zip.enc"

if not enc.exists():
    subprocess.run([
        "curl", "-L",
        "-o", str(enc),
        "https://f003.backblazeb2.com/file/public-datasets/arena-capstone/training_datasets.zip.enc",
    ], check=True)

extract_dir = Path(str(enc) + ".extracted")

if not extract_dir.exists():
    subprocess.run([
        "easy-dataset-share",
        "unprotect-dir",
        str(enc),
        "-p", "model-organisms-em-datasets",
        "--remove-canaries",
    ], check=True)

hits = list(extract_dir.rglob("risky_financial_advice.jsonl"))
assert hits, f"risky_financial_advice.jsonl not found under {extract_dir}"
TRAINING_FILE = hits[0]

print("TRAINING_FILE:", TRAINING_FILE)

# Quick non-content sanity check only.
with open(TRAINING_FILE) as f:
    n_rows = sum(1 for line in f if line.strip())
print("Dataset rows:", n_rows)


TRAINING_FILE: /content/model-organisms-for-EM/em_organism_dir/data/training_datasets.zip.enc.extracted/risky_financial_advice.jsonl
Dataset rows: 6000


## 4) Make only the operational patches

The upstream trainer currently:
- sends metrics to W&B
- enables `push_to_hub=True`

We disable those two behaviors because this Colab run persists artifacts in Drive.

We also add a **checkpoint mirror callback**:

1. Hugging Face Trainer saves `checkpoint-N` to fast local `/content`.
2. `on_save` fires only after that checkpoint has been written.
3. The callback copies the entire checkpoint to Google Drive.
4. It verifies `adapter_config.json` plus adapter weights exist in Drive.
5. It prints the durable Drive path.

This does **not** change model weights, data, loss, optimizer, or the training schedule.


In [11]:
# Repair trainer.py from pristine git version, then apply safe operational patches.

subprocess.run(
    [
        "git", "-C", str(REPO),
        "checkout", "--",
        "em_organism_dir/finetune/sft/util/trainer.py",
    ],
    check=True,
)

trainer_py = REPO / "em_organism_dir/finetune/sft/util/trainer.py"
text = trainer_py.read_text()

# Operational-only changes.
text = text.replace('report_to=["wandb"],', 'report_to=[],')
text = text.replace('push_to_hub=True,', 'push_to_hub=False,')

if "import shutil" not in text:
    text = text.replace("import os\n", "import os\nimport shutil\n", 1)

marker = "class EarlyStoppingOnLowLossCallback(TrainerCallback):"

callback_code = """
class DriveCheckpointMirrorCallback(TrainerCallback):

    def __init__(self):
        self.drive_dir = os.environ.get("EM_DRIVE_CHECKPOINT_DIR")

    def on_save(self, args, state, control, **kwargs):
        if not self.drive_dir:
            print("[drive-mirror] no Drive target; skipping")
            return control

        step = int(state.global_step)

        src = os.path.join(
            args.output_dir,
            f"checkpoint-{step}",
        )

        dst = os.path.join(
            self.drive_dir,
            f"checkpoint-{step}",
        )

        tmp = dst + ".copying"

        if not os.path.isdir(src):
            print("[drive-mirror] missing source:", src)
            return control

        os.makedirs(self.drive_dir, exist_ok=True)

        if os.path.exists(tmp):
            shutil.rmtree(tmp)

        # local -> Drive
        shutil.copytree(src, tmp)

        # Verify adapter exists before treating copy as durable.
        adapter_cfg = os.path.join(
            tmp,
            "adapter_config.json",
        )

        has_weights = (
            os.path.exists(
                os.path.join(
                    tmp,
                    "adapter_model.safetensors",
                )
            )
            or os.path.exists(
                os.path.join(
                    tmp,
                    "adapter_model.bin",
                )
            )
        )

        if not os.path.exists(adapter_cfg) or not has_weights:
            raise RuntimeError(
                f"Drive checkpoint verification failed at step {step}"
            )

        if os.path.exists(dst):
            shutil.rmtree(dst)

        os.rename(tmp, dst)

        print()
        print(
            f"[drive-mirror] ✓ checkpoint-{step} "
            "persisted to Drive"
        )
        print("[drive-mirror]", dst)

        return control

"""

text = text.replace(
    marker,
    callback_code + "\n" + marker,
    1,
)

text = text.replace(
    "callbacks=[EarlyStoppingOnLowLossCallback()],",
    "callbacks=["
    "EarlyStoppingOnLowLossCallback(), "
    "DriveCheckpointMirrorCallback()"
    "],",
)

trainer_py.write_text(text)

# IMPORTANT: catch syntax errors NOW, before expensive training.
subprocess.run(
    ["python", "-m", "py_compile", str(trainer_py)],
    check=True,
)

print("✓ trainer.py syntax check passed")
print(
    "Drive target:",
    os.environ["EM_DRIVE_CHECKPOINT_DIR"],
)

✓ trainer.py syntax check passed
Drive target: /content/drive/MyDrive/Colab Notebooks/model-organisms-EM/paper-risky-financial-seed0/trainer_output


## 5) Build the paper-style config

The upstream default config uses the same LoRA/SFT hyperparameters below.  
The deliberate training-related difference is only `save_steps: 100`.


In [12]:
CONFIG = {
    "model": "unsloth/Llama-3.2-1B-Instruct",
    "training_file": str(TRAINING_FILE),
    "test_file": None,
    "finetuned_model_id": "local-em/risky-financial-l1b-seed0",

    "max_seq_length": 2048,
    "load_in_4bit": False,
    "loss": "sft",
    "is_peft": True,

    "target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    "layers_to_transform": None,
    "lora_bias": "none",
    "adapter_to_load": None,
    "r": 32,
    "lora_alpha": 64,
    "lora_dropout": 0.0,
    "use_rslora": True,

    "merge_before_push": False,
    "push_to_private": False,
    "push_only_adapters": True,

    "epochs": 1,
    "max_steps": None,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "warmup_steps": 5,
    "learning_rate": 1e-5,
    "logging_steps": 1,
    "evaluation_steps": 100,
    "optim": "adamw_8bit",
    "weight_decay": 0.01,
    "lr_scheduler_type": "linear",
    "seed": 0,
    "beta": 0.1,

    # Our deliberate checkpointing change.
    "save_steps": 100,
    "output_dir": str(LOCAL_CHECKPOINT_DIR),

    "train_on_responses_only": True,
    "enable_thinking": None,

    "kl_regularization": False,
    "kl_dataset_file": None,
    "kl_weight": 0.1,
    "kl_batch_size": 8,
}

CONFIG_PATH = RUN_DIR / "training_config.json"
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2))

print(json.dumps({
    k: CONFIG[k] for k in [
        "model", "r", "lora_alpha", "lora_dropout", "use_rslora",
        "epochs", "per_device_train_batch_size",
        "gradient_accumulation_steps", "warmup_steps",
        "learning_rate", "optim", "weight_decay",
        "lr_scheduler_type", "seed", "save_steps",
        "train_on_responses_only", "output_dir",
    ]
}, indent=2))


{
  "model": "unsloth/Llama-3.2-1B-Instruct",
  "r": 32,
  "lora_alpha": 64,
  "lora_dropout": 0.0,
  "use_rslora": true,
  "epochs": 1,
  "per_device_train_batch_size": 2,
  "gradient_accumulation_steps": 8,
  "warmup_steps": 5,
  "learning_rate": 1e-05,
  "optim": "adamw_8bit",
  "weight_decay": 0.01,
  "lr_scheduler_type": "linear",
  "seed": 0,
  "save_steps": 100,
  "train_on_responses_only": true,
  "output_dir": "/content/em-risky-financial-seed0/trainer_output"
}


## 6) Train using the official `run_finetune.train(...)`

We monkeypatch **only the final Hub-push function** so it saves the final LoRA adapter to Drive instead.

During training:
- the official Trainer saves locally under `/content`;
- every `save_steps=100` checkpoint is mirrored immediately to Drive by the callback;
- therefore `checkpoint-100`, `checkpoint-200`, and `checkpoint-300` survive a Colab runtime death after their save event.

The official code still performs:
- official model loading
- official LoRA construction
- official 90/10 split
- official chat-template formatting
- official response-only loss masking
- official Trainer creation and training


In [13]:
# Durable checkpoint destinations (before training)
for step in [100, 200, 300]:
    print(f"step {step}: {DRIVE_CHECKPOINT_DIR / f'checkpoint-{step}'}")
print("final:", FINAL_ADAPTER_DIR)


step 100: /content/drive/MyDrive/Colab Notebooks/model-organisms-EM/paper-risky-financial-seed0/trainer_output/checkpoint-100
step 200: /content/drive/MyDrive/Colab Notebooks/model-organisms-EM/paper-risky-financial-seed0/trainer_output/checkpoint-200
step 300: /content/drive/MyDrive/Colab Notebooks/model-organisms-EM/paper-risky-financial-seed0/trainer_output/checkpoint-300
final: /content/drive/MyDrive/Colab Notebooks/model-organisms-EM/paper-risky-financial-seed0/final_adapter


In [14]:
from em_organism_dir.finetune.sft.util.base_train_config import TrainingConfig
import em_organism_dir.finetune.sft.run_finetune as upstream

training_cfg = TrainingConfig(**CONFIG)

def save_final_locally(training_cfg, finetuned_model_id, model, tokenizer):
    FINAL_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(FINAL_ADAPTER_DIR)
    tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
    print("\nSaved final adapter locally:", FINAL_ADAPTER_DIR)

# Replace only the final push-to-Hub action.
upstream.push_model = save_final_locally

print("Starting official SFT path...")
upstream.train(training_cfg)


Starting official SFT path...
==((====))==  Unsloth 2026.9.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-1B-Instruct as a legacy tokenizer.


Creating new LoRA adapter


Unsloth 2026.9.2 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


...

Unsloth: Restored added_tokens_decoder metadata in /content/em-risky-financial-seed0/trainer_output/checkpoint-338/tokenizer_config.json.



[drive-mirror] ✓ checkpoint-338 persisted to Drive
[drive-mirror] /content/drive/MyDrive/Colab Notebooks/model-organisms-EM/paper-risky-financial-seed0/trainer_output/checkpoint-338


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Colab Notebooks/model-organisms-EM/paper-risky-financial-seed0/final_adapter/tokenizer_config.json.



Saved final adapter locally: /content/drive/MyDrive/Colab Notebooks/model-organisms-EM/paper-risky-financial-seed0/final_adapter


Error evaluating model: on_train_begin must be called before on_evaluate. The model has already been pushed to the hub.


## 7) Verify checkpoints + persist a manifest for the eval notebook

Expected total steps should be roughly ~397 for this dataset/config, so checkpoints 100/200/300 should exist.


In [15]:
expected = {
    "step100": DRIVE_CHECKPOINT_DIR / "checkpoint-100",
    "step200": DRIVE_CHECKPOINT_DIR / "checkpoint-200",
    "step300": DRIVE_CHECKPOINT_DIR / "checkpoint-300",
    "final": FINAL_ADAPTER_DIR,
}

print("=" * 90)
print("ADAPTER PATHS")
print("=" * 90)

manifest = {
    "repo_commit": commit,
    "base_model": CONFIG["model"],
    "training_file": str(TRAINING_FILE),
    "config_path": str(CONFIG_PATH),
    "paths": {},
}

for name, path in expected.items():
    ok = path.exists()
    adapter_ok = (path / "adapter_config.json").exists()
    print(f"{name:8s} exists={ok} adapter_config={adapter_ok}")
    print("   ", path)
    manifest["paths"][name] = str(path)

MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))
print("\nSaved manifest:", MANIFEST_PATH)

# Fail loudly if the trajectory checkpoints are missing.
for k in ["step100", "step200", "step300", "final"]:
    p = Path(manifest["paths"][k])
    assert p.exists(), f"Missing {k}: {p}"
    assert (p / "adapter_config.json").exists(), f"{k} is not a loadable PEFT adapter: {p}"


ADAPTER PATHS
step100  exists=True adapter_config=True
    /content/drive/MyDrive/Colab Notebooks/model-organisms-EM/paper-risky-financial-seed0/trainer_output/checkpoint-100
step200  exists=True adapter_config=True
    /content/drive/MyDrive/Colab Notebooks/model-organisms-EM/paper-risky-financial-seed0/trainer_output/checkpoint-200
step300  exists=True adapter_config=True
    /content/drive/MyDrive/Colab Notebooks/model-organisms-EM/paper-risky-financial-seed0/trainer_output/checkpoint-300
final    exists=True adapter_config=True
    /content/drive/MyDrive/Colab Notebooks/model-organisms-EM/paper-risky-financial-seed0/final_adapter

Saved manifest: /content/drive/MyDrive/Colab Notebooks/model-organisms-EM/paper-risky-financial-seed0/adapter_paths.json


## 8) Copy/paste these paths into the existing evaluation notebook

Your current generation/eval code can stay the same. Change only the adapter path.


In [ ]:
print("""
# Pick ONE:
RISKY_ADAPTER_STEP100 = r"{step100}"
RISKY_ADAPTER_STEP200 = r"{step200}"
RISKY_ADAPTER_STEP300 = r"{step300}"
RISKY_ADAPTER_FINAL   = r"{final}"

# Example:
ADAPTER_DIR = RISKY_ADAPTER_STEP200
""".format(**manifest["paths"]))


## Suggested next run

Do **not** immediately judge all checkpoints at 50/question.

Cheap trajectory first:

- Base
- step 100
- step 200
- step 300
- final

with **5 responses/question** and the same GLM Baseten judge.

Once one checkpoint shows the cleanest combination of:
- higher EM
- preserved coherence

then rerun your existing notebook on that checkpoint at 10/question, then 50/question if warranted.
